In [ ]:
import pandas as pd
import numpy as np
import pickle
import torch
import torch.nn as nn
import re

import sys
sys.path.append('src')

# Configuração
PATH_DATASET_TESTE = "data/subm1.csv"
df_teste = pd.read_csv(PATH_DATASET_TESTE, sep=";")

# Carregar o LabelEncoder usado no treino
with open("modelo_numpy_artefactos.pkl", "rb") as f:
    artefactos = pickle.load(f)
le = artefactos["label_encoder"]

In [ ]:
print("-> A executar Inferência com Modelo NumPy...")
vectorizer = artefactos["vectorizer"]
net_np = artefactos["model"]

def clean_text_np(text):
    text = str(text).lower()
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    return text.strip()

X_teste_tfidf = vectorizer.transform(df_teste["Text"].apply(clean_text_np)).toarray()
preds_np_probs = net_np.predict(X_teste_tfidf)
preds_np_idx = np.argmax(preds_np_probs, axis=1)

df_teste["Predict_Numpy_DNN"] = le.inverse_transform(preds_np_idx)

In [ ]:
print("-> A executar Inferência com Modelo PyTorch (GRU)...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open("pytorch_vocab.pkl", "rb") as f:
    vocab = pickle.load(f)

class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        embedded = self.embedding(x)
        _, hidden = self.gru(embedded)
        hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        return self.fc(self.dropout(hidden))

model_pt = GRUClassifier(len(vocab), 128, 128, len(le.classes_)).to(device)
model_pt.load_state_dict(torch.load("modelo_pytorch_gru.pth", map_location=device))
model_pt.eval()

def encode(text, vocab, max_len=100):
    ids = [vocab.get(token, vocab["<unk>"]) for token in clean_text_np(text)][:max_len]
    if len(ids) < max_len:
        ids += [vocab["<pad>"]] * (max_len - len(ids))
    return ids

preds_pt_idx = []
with torch.no_grad():
    for text in df_teste["Text"].values:
        x = torch.tensor([encode(text, vocab)], dtype=torch.long).to(device)
        output = model_pt(x)
        pred = torch.argmax(output, dim=1).item()
        preds_pt_idx.append(pred)

df_teste["Predict_PyTorch_GRU"] = le.inverse_transform(preds_pt_idx)

In [ ]:
NOME_FICHEIRO_SAIDA = "resultados_grupoX.csv"  # Substituam 'X' pelo número do vosso grupo
df_teste.to_csv(NOME_FICHEIRO_SAIDA, index=False, sep=";")
print(f"Previsões finalizadas e guardadas em '{NOME_FICHEIRO_SAIDA}'!")
df_teste.head()